<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/07b_runner_loop_guards.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reading 07b — Can the Runner Get Stuck in a Loop?

> **A short reading, not a module** (about 15 minutes). It answers one question that comes up right after M07: *how much do I have to worry about the Runner looping forever, and how much of that does ADK already handle for me?*
>
> **No API key needed.** Every demo below runs for free: a stand-in model plays the part of a misbehaving LLM, so the runs cost nothing and give the same result every time.

> **Where you are** — you have seen the Runner loop in M01 (model call → tool → model call … until the model answers without asking for a tool), `max_iterations` on a `LoopAgent` in M05, and callbacks in M07. This reading connects the three.

# The Worry

Picture the help-desk agent from earlier modules. A user asks "where is order A-17?", the agent calls the warehouse tool, and the tool answers *"status: pending, try again later"*. A helpful model does exactly what it was told: it tries again. And again.

In M01 you saw that the Runner has no "done" counter of its own. It repeats **model call → tool → model call** until the model answers *without* asking for a tool. So the real question is not "can it loop?" — it can — but **"what stops it when the model never decides to stop?"**

There are three honest answers, and this reading walks through them:

1. **The model itself.** Most of the time a real model gives up after a few tries. Most of the time.
2. **ADK's fuse.** A hard limit on model calls that always fires — but late, and with an exception.
3. **Your own guard.** Ten lines of callback code that stop the turn politely. Only if you write them.

# What ADK Does For You (and What It Doesn't)

Everything below was checked against the source of `google-adk` 2.7.1, the version this course pins.

- **A fuse on model calls.** `RunConfig(max_llm_calls=...)` caps the number of model calls in one `runner.run_async(...)` turn. The default is **500**. The call that goes over the limit raises `LlmCallsLimitExceededError` straight out of `run_async` — the turn dies with an exception, there is no polite final message. It is one number for the whole run, not per agent.
- **A crashing tool is not a loop.** If your tool *raises* an exception, the exception comes out of `run_async` and the run stops at the first crash. Loops come from tools that *return* something the model reads as "try again".
- **`max_iterations` on a `LoopAgent`.** Default `None`, which means "loop until a child calls `exit_loop`". M05's rule stands: always set it. It counts loop rounds, not model calls, and every nested loop needs its own.
- **A retry helper you can switch on.** `ReflectAndRetryToolPlugin(max_retries=3)` catches tool *exceptions*, hands the model a structured "this failed, do not repeat the same call" message, and raises once the limit is used up. It does not see tools that report errors as a normal return value.

And what ADK does **not** have (as of 2.7.1): no "same tool called N times in a row" detector, no per-tool timeout, no token or money budget, no per-agent limit. Do the arithmetic on the default: 500 model calls at even half a cent each is \$2.50 and several minutes of waiting before the fuse blows. **The fuse protects your wallet from a disaster. It does not protect your user from a bad turn.**

### One picture

<pre>
user message
    │
    ▼
┌─► MODEL CALL  ◄── the fuse counts these (max_llm_calls)
│      │
│   asked for a tool?
│      │ no ──► final answer, the turn ends
│      │ yes
│      ▼
│   RUN TOOL  ◄── crash here = exception, run stops
│      │
└── result goes back to the model
</pre>

Nothing in this picture says "stop after N rounds" except the fuse. That is the whole story; the demos make it concrete.

# Setup

Install and imports only. **No API key this time** — the model in this notebook is a stand-in written in Python.

In [1]:
!pip install -q google-adk==2.7.1 litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null
print("✅ Packages installed.")

✅ Packages installed.


## Imports

Three new names next to the usual ones: `RunConfig` (where the fuse lives), `LlmCallsLimitExceededError` (what the fuse throws), and `BaseLlm` (the class every model plugs into — we need it for our stand-in).

In [2]:
import os
import sys, warnings, asyncio, uuid, logging
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")
import nest_asyncio; nest_asyncio.apply()
os.environ.setdefault("LITELLM_LOG", "ERROR")
# ADK logs a full traceback for every exception it re-raises; we print the exceptions ourselves below.
logging.getLogger("google_adk").setLevel(logging.CRITICAL)

from google.adk.agents import LlmAgent
from google.adk.agents.run_config import RunConfig
from google.adk.agents.invocation_context import LlmCallsLimitExceededError
from google.adk.runners import Runner
from google.adk.apps import App
from google.adk.sessions import InMemorySessionService
from google.adk.models.base_llm import BaseLlm
from google.adk.models.llm_response import LlmResponse
from google.adk.plugins import ReflectAndRetryToolPlugin
from google.genai import types

print("✅ Imports successful.")

✅ Imports successful.


# A Model That Never Stops

To *see* the fuse fire we need a model that misbehaves reliably. Real models misbehave only sometimes, which makes a bad demo. So we write a stand-in.

```python
class StubbornModel(BaseLlm):
```

You do not need to understand the class syntax. Read it as: *a model, in ADK's eyes* — it fits the same `model=` slot where you put `LiteLlm("openrouter/...")` in every other module. Its only behaviour: whatever you send, it answers "call `check_order_status`". A real model does this on a bad day. Ours does it every day.

*Optional detail, only if you're curious:* ADK asks a model for an answer through `generate_content_async`, which `yield`s one `LlmResponse`. Ours yields a response whose single part is a `function_call` — the same Part type you saw in M02's event stream.

In [3]:
class StubbornModel(BaseLlm):
    """A stand-in model: it ALWAYS asks for one tool and never gives a final answer."""
    model: str = "stubborn-model"
    tool_name: str

    async def generate_content_async(self, llm_request, stream=False):
        yield LlmResponse(content=types.Content(role="model", parts=[
            types.Part(function_call=types.FunctionCall(
                name=self.tool_name, args={"order_id": "A-17"}))
        ]))


TOOL_RUNS = {"count": 0}   # so we can count how often the tool really ran

def check_order_status(order_id: str) -> dict:
    """Look up the shipping status of an order in the warehouse system.

    Use this when a customer asks where their order is. Returns the current
    status; 'pending' means the warehouse has not confirmed dispatch yet.
    """
    TOOL_RUNS["count"] += 1
    return {"order_id": order_id, "status": "pending", "note": "try again later"}

print("✅ StubbornModel and check_order_status defined.")

✅ StubbornModel and check_order_status defined.


## The `chat()` Helper Again

Same helper as every module, with two small additions: it passes a `run_config` through to `run_async`, and it can build the Runner from an `App` when we hand it plugins (Demo 3). Run it and move on.

In [4]:
APP = "m07b_loop_guards"
USER = "student"
session_service = InMemorySessionService()

async def chat(agent, prompt: str, run_config=None, plugins=None):
    sid = f"s-{uuid.uuid4().hex[:6]}"
    await session_service.create_session(app_name=APP, user_id=USER, session_id=sid)
    if plugins:
        app = App(name=APP, root_agent=agent, plugins=plugins)
        runner = Runner(app=app, session_service=session_service)
    else:
        runner = Runner(agent=agent, app_name=APP, session_service=session_service)
    msg = types.Content(role="user", parts=[types.Part(text=prompt)])
    print(f"USER: {prompt}")
    TOOL_RUNS["count"] = 0
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg,
                                     run_config=run_config):
        if ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.function_call:
                    args = dict(p.function_call.args) if p.function_call.args else {}
                    print(f"  [tool_call] {p.function_call.name}({args})")
                if p.function_response:
                    print(f"  [tool_resp] {str(p.function_response.response)[:110]}")
                if p.text and p.text.strip():
                    tag = "[FINAL]" if ev.is_final_response() else "[step]"
                    print(f"  {tag} {ev.author}: {p.text.strip()}")
    print(f"  (turn finished normally; the tool ran {TOOL_RUNS['count']} times)\n")

print("✅ chat() ready.")

✅ chat() ready.


# Demo 1 — The Fuse: `max_llm_calls`

We want the run to stop even though the model never will. The fuse is one line:

```python
run_config=RunConfig(max_llm_calls=5)
```

`RunConfig` is a bag of per-run settings that travels with `run_async`; `max_llm_calls` is the one field that matters here. Five instead of the default 500 so we do not wait.

We wrap the call in `try/except`, because the fuse does not end the turn gently — it throws.

In [5]:
order_bot = LlmAgent(
    name="order_bot",
    model=StubbornModel(tool_name="check_order_status"),
    instruction="Answer questions about orders using the tools.",
    tools=[check_order_status],
)

try:
    await chat(order_bot, "Where is order A-17?", run_config=RunConfig(max_llm_calls=5))
except LlmCallsLimitExceededError as e:
    print(f"💥 LlmCallsLimitExceededError: {e}")
    print(f"   The turn died here. The tool ran {TOOL_RUNS['count']} times; the user got no answer.")

USER: Where is order A-17?


  [tool_call] check_order_status({'order_id': 'A-17'})
  [tool_resp] {'order_id': 'A-17', 'status': 'pending', 'note': 'try again later'}
  [tool_call] check_order_status({'order_id': 'A-17'})
  [tool_resp] {'order_id': 'A-17', 'status': 'pending', 'note': 'try again later'}
  [tool_call] check_order_status({'order_id': 'A-17'})
  [tool_resp] {'order_id': 'A-17', 'status': 'pending', 'note': 'try again later'}
  [tool_call] check_order_status({'order_id': 'A-17'})
  [tool_resp] {'order_id': 'A-17', 'status': 'pending', 'note': 'try again later'}
  [tool_call] check_order_status({'order_id': 'A-17'})
  [tool_resp] {'order_id': 'A-17', 'status': 'pending', 'note': 'try again later'}
💥 LlmCallsLimitExceededError: Max number of llm calls limit of `5` exceeded
   The turn died here. The tool ran 5 times; the user got no answer.


### 🔍 What just happened?

Five model calls, five tool calls, and the sixth model call was refused: `Max number of llm calls limit of 5 exceeded`. Notice where the message came from — the `except` block, not the agent. Nothing was printed by `chat()` after the fifth tool result, because the exception came out of `run_async` and ended the whole `chat()` call. The user is left with nothing.

With the default of 500 the picture is identical, only 100 times more expensive and several minutes later. That is what "fuse" means: it prevents a disaster, it does not produce a good turn.

⚠️ One gotcha worth knowing: a model call that a `before_model_callback` answers itself (M07's blocklist, for example) never reaches the real model — and never reaches this counter. The fuse only counts calls that actually go out.

### 🎯 Mini-task

Set `max_llm_calls=1` and run the cell again. How many `[tool_call]` lines do you see before the exception? Explain the number before you read on. (Hint: which call is the one that trips the fuse — the first model call, or the one *after* the tool result comes back?)

# Demo 2 — A Circuit Breaker in Ten Lines

We want something better than a crash: after three pointless tool calls the agent should **stop and tell the user**. Nothing in ADK does this, but two callbacks from M07 are enough:

- a **`before_tool_callback`** that counts calls per tool in session state,
- a **`before_model_callback`** that checks the count and, once the budget is spent, returns a final text answer instead of calling the model.

The second one is M07's return-to-override rule doing new work: the `LlmResponse` we return has *no* tool call in it, so the Runner treats it as the final answer and the turn ends normally — no exception.

```python
tool_context.state[f"temp:calls:{tool.name}"] = ...
```

One detail in the counter: the `temp:` prefix. State keys that start with `temp:` live only for the current turn, so the budget resets with every new user message — without it, three calls on Monday would block the agent on Tuesday.

In [6]:
TOOL_CALL_BUDGET = 3

def count_tool_calls(tool, args, tool_context):
    key = f"temp:calls:{tool.name}"
    tool_context.state[key] = tool_context.state.get(key, 0) + 1
    return None                      # None = let the tool run

def circuit_breaker(callback_context, llm_request):
    used = sum(v for k, v in callback_context.state.to_dict().items()
               if k.startswith("temp:calls:"))
    if used >= TOOL_CALL_BUDGET:
        return LlmResponse(content=types.Content(role="model", parts=[types.Part(
            text=f"I stopped after {used} tool calls without a usable result. "
                 f"Please check order status manually or try again later.")]))
    return None                      # None = call the model as usual

order_bot_safe = LlmAgent(
    name="order_bot_safe",
    model=StubbornModel(tool_name="check_order_status"),
    instruction="Answer questions about orders using the tools.",
    tools=[check_order_status],
    before_tool_callback=count_tool_calls,
    before_model_callback=circuit_breaker,
)

await chat(order_bot_safe, "Where is order A-17?", run_config=RunConfig(max_llm_calls=50))

USER: Where is order A-17?
  [tool_call] check_order_status({'order_id': 'A-17'})
  [tool_resp] {'order_id': 'A-17', 'status': 'pending', 'note': 'try again later'}
  [tool_call] check_order_status({'order_id': 'A-17'})
  [tool_resp] {'order_id': 'A-17', 'status': 'pending', 'note': 'try again later'}
  [tool_call] check_order_status({'order_id': 'A-17'})
  [tool_resp] {'order_id': 'A-17', 'status': 'pending', 'note': 'try again later'}
  [FINAL] order_bot_safe: I stopped after 3 tool calls without a usable result. Please check order status manually or try again later.
  (turn finished normally; the tool ran 3 times)



### 🔍 What just happened?

Three tool calls, then a `[FINAL]` message — and the last line says the turn *finished normally*. The message was written by `circuit_breaker`, not by the model: on the fourth model call the callback saw `used == 3`, returned its own `LlmResponse`, and the Runner accepted it as the final answer. The fuse at 50 was never touched.

Compare the two demos once more. Same stubborn model, same tool. Demo 1 ends in an exception your web app has to catch and explain. Demo 2 ends in a sentence the user can read. Ten lines of difference.

### 🎯 Mini-task

1. Make the budget **per tool**. The simplest place is `before_tool_callback` itself: once `temp:calls:{tool.name}` reaches the budget, return a dict such as `{"error": "budget exhausted, stop calling this tool"}` instead of `None` — the tool is skipped and the model receives your dict as its result.
2. Swap `StubbornModel(...)` for your usual `LiteLlm("openrouter/openai/gpt-5.6-luna")` and load your OpenRouter key as in M07. Does the real model ever reach the budget with the "pending, try again later" tool? Run it five times — one run proves nothing.

# Demo 3 — Crashes, and the Retry Plugin

Last worry: the tool does not loop, it **crashes** — the warehouse API times out and the function raises. What does the Runner do? Let's look, without any protection first.

In [7]:
def check_order_status_flaky(order_id: str) -> dict:
    """Look up the shipping status of an order (backend is unreliable today)."""
    TOOL_RUNS["count"] += 1
    raise ConnectionError("warehouse API timed out")

order_bot_flaky = LlmAgent(
    name="order_bot_flaky",
    model=StubbornModel(tool_name="check_order_status_flaky"),
    instruction="Answer questions about orders using the tools.",
    tools=[check_order_status_flaky],
)

try:
    await chat(order_bot_flaky, "Where is order A-17?", run_config=RunConfig(max_llm_calls=10))
except Exception as e:
    print(f"💥 {type(e).__name__}: {e}")
    print(f"   The tool ran {TOOL_RUNS['count']} time(s). No loop — the first crash ended the run.")

USER: Where is order A-17?
  [tool_call] check_order_status_flaky({'order_id': 'A-17'})
💥 ConnectionError: warehouse API timed out
   The tool ran 1 time(s). No loop — the first crash ended the run.


### 🔍 What just happened?

One tool call, one `ConnectionError`, run over. The exception travelled from inside your tool, through the Runner, out of `run_async`. ADK deliberately does **not** swallow tool exceptions and feed them to the model — the maintainers' reasoning is that a raw exception text can leak internal details to the model (and thus to the user).

So a crashing tool is the opposite of a loop. If you *want* the model to get a second chance, you ask for it explicitly. That is what the plugin is for.

## Switching on `ReflectAndRetryToolPlugin`

```python
app = App(name=APP, root_agent=agent, plugins=[ReflectAndRetryToolPlugin(max_retries=2)])
runner = Runner(app=app, session_service=session_service)
```

Two new things, read inside-out. `ReflectAndRetryToolPlugin(max_retries=2)` is a ready-made piece of ADK code that catches tool exceptions and, instead of crashing, sends the model a structured "this failed, here is why, do not repeat the exact same call" message — up to two times. And `App(...)` is where plugins live: the `Runner(agent=..., app_name=...)` form you have used all course is the short version of `Runner(app=App(name=..., root_agent=...))`. Same Runner, one wrapper more.

Our `chat()` helper does this wrapping when you pass `plugins=[...]`.

In [8]:
try:
    await chat(order_bot_flaky, "Where is order A-17?",
               run_config=RunConfig(max_llm_calls=10),
               plugins=[ReflectAndRetryToolPlugin(max_retries=2)])
except Exception as e:
    print(f"💥 {type(e).__name__}: {str(e)[:120]}")
    print(f"   The tool ran {TOOL_RUNS['count']} times: one try plus two retries, then the plugin gave up.")

USER: Where is order A-17?
  [tool_call] check_order_status_flaky({'order_id': 'A-17'})
  [tool_resp] {'response_type': 'ERROR_HANDLED_BY_REFLECT_AND_RETRY_PLUGIN', 'error_type': 'ConnectionError', 'error_details
  [tool_call] check_order_status_flaky({'order_id': 'A-17'})
  [tool_resp] {'response_type': 'ERROR_HANDLED_BY_REFLECT_AND_RETRY_PLUGIN', 'error_type': 'ConnectionError', 'error_details
  [tool_call] check_order_status_flaky({'order_id': 'A-17'})
💥 RuntimeError: Error in plugin 'reflect_retry_tool_plugin' during 'on_tool_error_callback' callback: warehouse API timed out
   The tool ran 3 times: one try plus two retries, then the plugin gave up.


### 🔍 What just happened?

Three tool calls instead of one. After each crash the model received a `[tool_resp]` that starts with `ERROR_HANDLED_BY_REFLECT_AND_RETRY_PLUGIN` — that dict carries the error type, the arguments used, and a paragraph of guidance ("analyze the error… do not repeat the exact same call"). A real model reads that and changes its arguments or gives up. Our stubborn one repeats itself, so after retry 2 of 2 the plugin raised and the run ended — bounded, exactly as configured.

⚠️ The plugin only sees **exceptions**. A tool that returns `{"status": "error"}` as a normal value looks like success to it and is retried by nobody but the model. Decide per tool: raise on real failures (the plugin bounds the retries), return plain data for expected outcomes like "not found" — and word that data so it sounds final, not like an invitation to try again.

# What People Actually Hit (a Forum Digest)

A read through the `google/adk-python` GitHub issues and discussions (checked 2026-09-15). The recurring loop reports, in plain words:

- **Forced tool use.** `FunctionCallingConfig(mode="ANY")` tells the model it *must* call a tool every time — so it does, forever, even when the previous result was enough ([#4179](https://github.com/google/adk-python/issues/4179)). Forcing removes the model's only way to say "done".
- **`output_schema` together with `tools`.** The model is asked to end with a structured object *and* may call tools; several reports of the tool being called dozens of times before the fuse ([#3413](https://github.com/google/adk-python/issues/3413)). Split it: tools in one agent, the structured summary in a second agent after it (M05).
- **A crashing tool takes the whole multi-agent run down.** Exactly Demo 3. The maintainers' answer in [discussion #795](https://github.com/google/adk-python/discussions/795): tool authors should return structured error dicts (or use the plugin) rather than expecting ADK to catch everything.
- **`LoopAgent` not stopping.** `exit_loop` called but the loop continuing ([#2988](https://github.com/google/adk-python/issues/2988)); nested loops where one child's exit ends the outer loop too ([#2808](https://github.com/google/adk-python/issues/2808), [#1376](https://github.com/google/adk-python/issues/1376)). The lesson is M05's: `max_iterations` on *every* loop, never trust the exit signal alone. (Note that in ADK 2.7 `LoopAgent` also carries a deprecation note in favour of the newer Workflow runtime; it still works, and the rule is the same.)
- **Version regressions.** A code-executor loop that was fine in 1.16 and broken from 1.18 ([#3921](https://github.com/google/adk-python/issues/3921)). "The framework handles it" is true for a version, not forever — one more reason the course pins its versions.
- **Local models.** From this course's own notes: with Ollama, the `ollama/` model prefix causes endless tool-call loops; `ollama_chat/` does not.

The pattern across all of them: almost every loop is the model being **given a reason to call again** — forced tool use, an error handed back as data, history it cannot see — rather than ADK losing its stop condition. Fix the reason first; then bound the damage.

# The Answer

So, how much do you have to think about it, and can you rely on ADK? A rule of thumb in three tiers.

**Rely on ADK for the fuse — but set it yourself.** Put `run_config=RunConfig(max_llm_calls=20)` (or whatever a real turn of *your* agent needs, times four) on every `run_async` call. A help-desk turn needs one to five model calls; 500 is a number for nobody. And catch `LlmCallsLimitExceededError` where you call the Runner, the same way you catch a network error.

**Rely on ADK for loops you declared.** `max_iterations` on every `LoopAgent`, no exceptions. That is the framework's own mechanism; use it.

**Write your own for the user-facing turn — and keep it tiny.** Two things are worth your code:
1. **Tools that sound final.** Return `{"status": "not_found"}` or `{"status": "pending", "next_check": "tomorrow"}`; never "try again later". Raise on real failures and let `ReflectAndRetryToolPlugin` bound the retries.
2. **The ten-line circuit breaker** from Demo 2, whenever the agent talks to real users or the tools cost real money. Count in `temp:` state, end the turn with a sentence.

**Do not write:** your own `while` loop around `run_async`, retry loops *inside* tools that hide failures from the model, or a token counter unless you bill per user. Every one of these fights the Runner instead of using it.

And when you have the breaker, **test it** (M09): an eval case with a tool that answers "pending" forever belongs in your eval set, because the day the model gets stubborn is the day you are not watching.

# Key Takeaways

- **The Runner loops until the model answers without a tool call.** There is no other stop condition of its own.
- **`RunConfig(max_llm_calls=N)` is a fuse:** default 500 per turn, raises `LlmCallsLimitExceededError` out of `run_async`. Set it low, catch it.
- **Callback-answered model calls do not count** toward the fuse.
- **A crashing tool stops the run, it does not loop.** `ReflectAndRetryToolPlugin(max_retries=N)` (via `App(plugins=[...])`) turns that into a bounded number of retries — for exceptions only.
- **`max_iterations` on every `LoopAgent`.** Do not rely on `exit_loop` alone.
- **Your own guard is ten lines:** count tool calls in `temp:` state in `before_tool_callback`, end the turn from `before_model_callback` with a final `LlmResponse`.
- **Most real loops have a reason** — forced tool use, an error returned as data, a tool that says "try again". Fix the reason, then bound the damage.